# Análisis: Elecciones Lok Sabha India 2024
Este notebook realiza la limpieza, exploración y un modelo predictivo mínimo sobre el dataset de resultados electorales de India (2024).

**Instrucciones:** colocar el CSV original en la carpeta `data/` con el nombre `india_lok_sabha_election_results_2024.csv` o usar la API de Kaggle (ver celda de carga).

In [ ]:
# Recomendado: instalar dependencias si hace falta (ejecutar una vez)
# !pip install pandas numpy matplotlib seaborn scikit-learn kaggle openpyxl


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

sns.set(style="whitegrid")
%matplotlib inline

In [ ]:
# Cargar dataset desde data/ o mostrar instrucciones para descargar desde Kaggle
path = os.path.join('data', 'india_lok_sabha_election_results_2024.csv')
if os.path.exists(path):
    df = pd.read_csv(path, low_memory=False)
    print('Archivo cargado desde:', path)
else:
    print('Archivo no encontrado en data/.
Descarga manual (web) desde: https://www.kaggle.com/datasets/nuhmanpk/india-lok-sabha-election-results-2024')
    print('
Si prefieres usar la API de Kaggle:')
    print('1) Instala kaggle y configura tu API token (coloca kaggle.json en ~/.kaggle/).')
    print('2) Ejecuta: kaggle datasets download -d nuhmanpk/india-lok-sabha-election-results-2024 -p data --unzip')
    raise SystemExit('Coloca el CSV en data/ y vuelve a ejecutar la celda')

In [ ]:
# Vista rápida y estructura
df.head()

In [ ]:
# Comprobaciones iniciales
print('Filas, columnas:', df.shape)
print('
Tipos de datos:')
print(df.dtypes)
print('
Valores nulos por columna:')
print(df.isnull().sum().sort_values(ascending=False).head(20))

## 1) Limpieza de datos
Realizaremos: eliminación de duplicados, normalización de textos categóricos, conversión de tipos, manejo de nulos y detección de anomalías.

In [ ]:
# 1. Detección y eliminación de duplicados
before = df.shape[0]
df = df.drop_duplicates(ignore_index=True)
after = df.shape[0]
print(f'Duplicados eliminados: {before - after}')

# 2. Normalizar columnas categóricas comunes si existen
for col in ['Party', 'party', 'PARTY']:
    if col in df.columns:
        df['Party'] = df[col].astype(str).str.strip().str.title()
        break
# Asegurarse de que exista columna 'Party'
if 'Party' not in df.columns and 'party_name' in df.columns:
    df['Party'] = df['party_name'].astype(str).str.strip().str.title()

# Normalizar columnas de texto genéricas
text_cols = df.select_dtypes(include=['object']).columns.tolist()
for c in text_cols:
    df[c] = df[c].astype(str).str.strip()

# 3. Conversión de tipos numéricos: buscar columnas que contengan 'vote' o 'votes' o 'total'
num_cols = []
for c in df.columns:
    if any(k in c.lower() for k in ['vote', 'votes', 'total', 'margin', 'turnout', 'electors']):
        # intentamos convertir a numérico
        try:
            df[c] = pd.to_numeric(df[c].astype(str).str.replace(',',''), errors='coerce')
            num_cols.append(c)
        except Exception:
            pass
print('Columnas numéricas detectadas y convertidas:', num_cols)

# 4. Manejo de valores faltantes: rellenar con marcadores apropiados
for c in df.columns:
    if df[c].dtype == 'O':
        df[c] = df[c].replace({'nan': None})
        df[c] = df[c].fillna('Unknown')
    else:
        # para numericos usar NaN (ya lo son) o  -1 como marcador en ciertos casos
        if df[c].isnull().sum() > 0:
            df[c] = df[c].fillna(-1)

# 5. Detección de anomalías básicas: votos mayores que electores/total_votes
possible_totals = [c for c in df.columns if any(k in c.lower() for k in ['total', 'electors', 'electors_no', 'voters'])]
print('Posibles columnas de total de electores/registro:', possible_totals)

# Ejemplo de regla: si existe columna 'Total Votes' o similar y columna 'Votes' comparar
vote_cols = [c for c in df.columns if 'vote' in c.lower() and c not in possible_totals]
print('Columnas detectadas como votos de candidato (heurística):', vote_cols)

if vote_cols and possible_totals:
    for v in vote_cols:
        for t in possible_totals:
            mask = df[v] > df[t]
            if mask.any():
                print(f'Anomalías detectadas: {mask.sum()} filas donde {v} > {t}. Se fijarán a NaN')
                df.loc[mask, v] = np.nan

# Resumen después de limpieza básica
print(df.shape)
print(df[num_cols].describe().T if num_cols else 'No hay columnas numéricas detectadas')

## 2) Visualizaciones exploratorias
A continuación se generan al menos 5 visualizaciones y una breve interpretación junto a cada una.

In [ ]:
# 1) Univariada: Histograma de una columna de votos (si existe)
if vote_cols:
    col = vote_cols[0]
    plt.figure(figsize=(8,5))
    sns.histplot(df[col].dropna(), bins=50, kde=False)
    plt.title(f'Histograma de {col}')
    plt.xlabel('Votos')
    plt.ylabel('Frecuencia')
    plt.show()
    print('Interpretación: muestra la distribución de los votos por candidato; cola larga indica candidatos con muchos votos.')
else:
    print('No se detectó una columna de votos para el histograma (según heurística).')

In [ ]:
# 2) Univariada: Top 10 partidos por número de candidaturas
if 'Party' in df.columns:
    top = df['Party'].value_counts().nlargest(10)
    plt.figure(figsize=(10,5))
    sns.barplot(x=top.values, y=top.index, palette='tab10')
    plt.title('Top 10 partidos por número de candidaturas')
    plt.xlabel('Número de candidaturas')
    plt.ylabel('Partido')
    plt.show()
    print('Interpretación: muestra la presencia relativa de los partidos en las candidaturas; partidos grandes aparecen arriba.')
else:
    print('Columna Party no encontrada.')

In [ ]:
# 3) Multivariada: Boxplot de votos por partido (solo top 8 partidos para legibilidad)
if 'Party' in df.columns and vote_cols:
    col = vote_cols[0]
    top_parties = df['Party'].value_counts().nlargest(8).index.tolist()
    plt.figure(figsize=(12,6))
    sns.boxplot(x='Party', y=col, data=df[df['Party'].isin(top_parties)])
    plt.xticks(rotation=45)
    plt.title('Distribución de votos por partido (top 8)')
    plt.show()
    print('Interpretación: compara la mediana y dispersión de votos entre los partidos más frecuentes.')
else:
    print('No hay datos suficientes para boxplot por partido.')

In [ ]:
# 4) Multivariada: Scatter votos vs total electores (si existe)
if vote_cols and possible_totals:
    v = vote_cols[0]
    t = possible_totals[0]
    plt.figure(figsize=(8,6))
    sns.scatterplot(x=df[t], y=df[v], alpha=0.6)
    plt.xlabel(t)
    plt.ylabel(v)
    plt.title(f'{v} vs {t}')
    plt.show()
    print('Interpretación: verifica relación entre tamaño del electorado y votos obtenidos por candidato.')
else:
    print('No se detectaron columnas para scatter votes vs total.')

In [ ]:
# 5) Matriz de correlación (numéricas)
if len(num_cols) >= 2:
    plt.figure(figsize=(8,6))
    corr = df[num_cols].corr()
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm')
    plt.title('Correlación entre variables numéricas')
    plt.show()
    print('Interpretación: identifica correlaciones fuertes entre variables numéricas.')
else:
    print('No hay suficientes columnas numéricas para matriz de correlación.')

## 3) Estadísticas descriptivas y correlación
Se presentan medidas de tendencia central y dispersión para variables numéricas detectadas.

In [ ]:
# Resumen estadístico
if num_cols:
    display(df[num_cols].describe().T)
    # adicional: mediana y desviación
    stats = df[num_cols].agg(['median','mean','std']).T
    display(stats)
else:
    print('No hay columnas numéricas para estadísticas.')

## 4) Modelado predictivo (clasificación)
Predeciremos si un registro/candidato fue ganador mediante un clasificador. Si no existe la etiqueta, la inferimos por mayor número de votos por circunscripción.

In [ ]:
# Preparar etiqueta `winner` si no existe
if 'winner' in df.columns or 'Winner' in df.columns:
    winner_col = 'winner' if 'winner' in df.columns else 'Winner'
    df['winner_flag'] = df[winner_col].astype(str).str.lower().isin(['y','yes','winner','won','true','1'])
else:
    # Heurística: por cada constituency, el candidato con mayor número de votos es el ganador
    key_const = None
    for k in ['Constituency', 'constituency', 'constituency_name']:
        if k in df.columns:
            key_const = k
            break
    if key_const and vote_cols:
        df['winner_flag'] = False
        idx = df.groupby(key_const)[vote_cols[0]].transform('max') == df[vote_cols[0]]
        df.loc[idx, 'winner_flag'] = True
    else:
        raise SystemExit('No se pudo inferir etiqueta winner. Se necesita columna de circunscripción y votos.')

# Elegir características simples
features = []
# usar columnas numéricas encontradas (limitar)
for c in num_cols:
    if c != vote_cols[0]:
        features.append(c)
# incluir votos del candidato como característica
features = [vote_cols[0]] + features
# incluir party codificada (top 10)
if 'Party' in df.columns:
    top = df['Party'].value_counts().nlargest(10).index.tolist()
    df['Party_mod'] = df['Party'].where(df['Party'].isin(top), 'Other')
    le = LabelEncoder()
    df['Party_enc'] = le.fit_transform(df['Party_mod'])
    features.append('Party_enc')

X = df[features].fillna(-1)
y = df['winner_flag'].astype(int)

# Entrenar modelo simple
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print('Accuracy:', accuracy_score(y_test, y_pred))
print('
Clasification report:')
print(classification_report(y_test, y_pred))

# Matriz de confusión
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicho')
plt.ylabel('Verdadero')
plt.title('Matriz de confusión')
plt.show()
print('Interpretación: revisar sensibilidad y precisión para la clase ganadora; verificar si hay desbalance.')

## 5) Conclusiones y pasos siguientes
- Resumen de hallazgos (completar tras ejecutar con el CSV real).
- Instrucciones para subir a GitHub y crear TAG:
  1) `git init` (si no hay repo)
  2) `git add election_analysis.ipynb requirements.txt`
  3) `git commit -m "Entrega: análisis elecciones India 2024"`
  4) Crear repo remoto en GitHub y `git remote add origin <url>`
  5) `git push -u origin main`
  6) Crear tag con fecha de entrega: `git tag -a v1.0-DELIVERY-2024-07-03 -m "Entrega"` y luego `git push origin --tags`.

**Nota:** reemplaza la fecha en el tag por la fecha real de entrega.